In [1]:
# 06_mlp_baseline.ipynb
# Train and evaluate a small MLP on Top-7 features (Keras / TensorFlow)

In [2]:
import os, json, time, joblib
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import matplotlib.pyplot as plt

# Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

%matplotlib inline

# Paths
DATA_PATH = "../data/heart_uci.csv"
FEATURE_MAP = "../artifacts/feature_index_map.json"
PREPROCESSOR_PICKLE = "../artifacts/preprocessor_for_training.joblib"  # optional; else reconstruct
ART_DIR = "../artifacts"
FIG_DIR = "../figures"
MODEL_OUT = os.path.join(ART_DIR, "mlp_top7.h5")
METRICS_OUT = os.path.join(ART_DIR, "mlp_metrics_k7.json")
os.makedirs(ART_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
print("Artifacts:", ART_DIR)

Artifacts: ../artifacts


In [4]:
df = pd.read_csv(DATA_PATH)
df['num'] = (df['num'] > 0).astype(int)
X_df = df.drop(columns=['num'])
y = df['num'].values

# use same holdout split as before (for consistent comparison)
X_train, X_hold, y_train, y_hold = train_test_split(X_df, y, test_size=0.2, random_state=42, stratify=y)
print("Train / Hold:", X_train.shape, X_hold.shape)

Train / Hold: (736, 15) (184, 15)


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

numeric_features = ['age','trestbps','chol','thalch','oldpeak','ca']
categorical_features = ['sex','cp','fbs','restecg','exang','slope','thal']

# categories list from full data (same as earlier)
categories_list = [sorted(X_df[c].dropna().unique().tolist()) for c in categorical_features]

numeric_transformer = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False, categories=categories_list))])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
], remainder="drop", sparse_threshold=0)

# fit on train only
preprocessor.fit(X_train)
# compute transformed names
def get_feature_names(ct):
    names=[]
    for name, transformer, cols in ct.transformers_:
        if name == "remainder": continue
        if hasattr(transformer, "named_steps"):
            last = list(transformer.named_steps.values())[-1]
            if hasattr(last, "get_feature_names_out"):
                out = last.get_feature_names_out(cols)
                names.extend(out.tolist())
            else:
                names.extend(list(cols))
        else:
            if hasattr(transformer, "get_feature_names_out"):
                names.extend(transformer.get_feature_names_out(cols).tolist())
            else:
                names.extend(list(cols))
    return names

full_feature_names = get_feature_names(preprocessor)
X_train_trans = preprocessor.transform(X_train)
X_hold_trans = preprocessor.transform(X_hold)
X_train_df = pd.DataFrame(X_train_trans, columns=full_feature_names, index=X_train.index)
X_hold_df  = pd.DataFrame(X_hold_trans, columns=full_feature_names, index=X_hold.index)

# load top7
meta = json.load(open(FEATURE_MAP))
top7 = meta["selected_features"]
print("Top7:", top7)

X_train_top7 = X_train_df[top7].to_numpy()
X_hold_top7  = X_hold_df[top7].to_numpy()

print("Shapes (MLP input):", X_train_top7.shape, X_hold_top7.shape)

Top7: ['cp_asymptomatic', 'exang_False', 'cp_atypical angina', 'chol', 'sex_Female', 'oldpeak', 'thal_normal']
Shapes (MLP input): (736, 7) (184, 7)


In [8]:
# Cell 4: small MLP architecture (tuneable)
def build_mlp(input_dim, hidden_layers=[64,32], dropout_rate=0.3):
    model = Sequential()
    # input + first hidden
    model.add(Dense(hidden_layers[0], input_dim=input_dim, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))
    # additional hidden
    if len(hidden_layers) > 1:
        for units in hidden_layers[1:]:
            model.add(Dense(units, activation='relu'))
            model.add(BatchNormalization())
            model.add(Dropout(dropout_rate))
    # output
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

model = build_mlp(input_dim=X_train_top7.shape[1], hidden_layers=[64,32], dropout_rate=0.25)
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                      │ (None, 64)                  │             512 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 64)                  │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 32)                  │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,009 (11.75 KB)

 Trainable params: 2,817 (11.00 KB)

 Non-trainable params: 192 (768.00 B)

In [10]:
# Cell 5: training (with early stopping)
batch_size = 32
epochs = 200
es = EarlyStopping(monitor='val_auc', mode='max', patience=20, restore_best_weights=True, verbose=1)
mc = ModelCheckpoint(MODEL_OUT, monitor='val_auc', mode='max', save_best_only=True, verbose=1)

history = model.fit(
    X_train_top7, y_train,
    validation_split=0.15,   # small internal validation
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[es, mc],
    verbose=2
)

Epoch 1/200

Epoch 1: val_auc improved from None to 0.87294, saving model to ../artifacts\mlp_top7.h5


20/20 - 1s - 36ms/step - auc: 0.8653 - loss: 0.4618 - val_auc: 0.8729 - val_loss: 0.4859
Epoch 2/200

Epoch 2: val_auc improved from 0.87294 to 0.87327, saving model to ../artifacts\mlp_top7.h5


20/20 - 1s - 53ms/step - auc: 0.8773 - loss: 0.4424 - val_auc: 0.8733 - val_loss: 0.4770
Epoch 3/200

Epoch 3: val_auc did not improve from 0.87327
20/20 - 1s - 35ms/step - auc: 0.8738 - loss: 0.4395 - val_auc: 0.8718 - val_loss: 0.4741
Epoch 4/200

Epoch 4: val_auc did not improve from 0.87327
20/20 - 1s - 34ms/step - auc: 0.8626 - loss: 0.4632 - val_auc: 0.8708 - val_loss: 0.4734
Epoch 5/200

Epoch 5: val_auc did not improve from 0.87327
20/20 - 0s - 25ms/step - auc: 0.8827 - loss: 0.4279 - val_auc: 0.8724 - val_loss: 0.4696
Epoch 6/200

Epoch 6: val_auc improved from 0.87327 to 0.87442, saving model to ../artifacts\mlp_top7.h5


20/20 - 1s - 38ms/step - auc: 0.8710 - loss: 0.4452 - val_auc: 0.8744 - val_loss: 0.4612
Epoch 7/200

Epoch 7: val_auc did not improve from 0.87442
20/20 - 1s - 26ms/step - auc: 0.8676 - loss: 0.4551 - val_auc: 0.8710 - val_loss: 0.4628
Epoch 8/200

Epoch 8: val_auc did not improve from 0.87442
20/20 - 0s - 20ms/step - auc: 0.8734 - loss: 0.4452 - val_auc: 0.8706 - val_loss: 0.4633
Epoch 9/200

Epoch 9: val_auc did not improve from 0.87442
20/20 - 0s - 23ms/step - auc: 0.8785 - loss: 0.4436 - val_auc: 0.8713 - val_loss: 0.4618
Epoch 10/200

Epoch 10: val_auc did not improve from 0.87442
20/20 - 0s - 21ms/step - auc: 0.8826 - loss: 0.4276 - val_auc: 0.8733 - val_loss: 0.4588
Epoch 11/200

Epoch 11: val_auc did not improve from 0.87442
20/20 - 1s - 51ms/step - auc: 0.8766 - loss: 0.4433 - val_auc: 0.8695 - val_loss: 0.4591
Epoch 12/200

Epoch 12: val_auc did not improve from 0.87442
20/20 - 2s - 85ms/step - auc: 0.8790 - loss: 0.4336 - val_auc: 0.8636 - val_loss: 0.4649
Epoch 13/200

Epo

In [12]:
# Cell 6: evaluate
best_model = tf.keras.models.load_model(MODEL_OUT)  # ensure best saved
probs_mlp = best_model.predict(X_hold_top7).ravel()
roc_mlp = roc_auc_score(y_hold, probs_mlp)
ap_mlp = average_precision_score(y_hold, probs_mlp)
brier_mlp = brier_score_loss(y_hold, probs_mlp)

mlp_metrics = {
    "model": "MLP",
    "features": "Top-7",
    "roc_auc": float(roc_mlp),
    "avg_precision": float(ap_mlp),
    "brier": float(brier_mlp),
    "train_epochs": int(len(history.history['loss']))
}
json.dump(mlp_metrics, open(METRICS_OUT, "w"), indent=2)
print("MLP metrics:", mlp_metrics)

6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step
MLP metrics: {'model': 'MLP', 'features': 'Top-7', 'roc_auc': 0.9226446676231468, 'avg_precision': 0.936820982006437, 'brier': 0.11817093154796587, 'train_epochs': 26}


In [15]:
# Cell 7: append to model comparison CSV (same as LR step)
COMPARE_CSV = os.path.join(ART_DIR, "model_comparison.csv")
row = {
    "model": "MLP",
    "features": "Top-7",
    "roc_auc": mlp_metrics["roc_auc"],
    "avg_precision": mlp_metrics["avg_precision"],
    "brier": mlp_metrics["brier"],
    "params": "hidden=[64,32],dropout=0.25,adam",
    "train_time_s": "N/A",
    "artifact": os.path.basename(MODEL_OUT)
}
import pandas as pd
if not os.path.exists(COMPARE_CSV):
    pd.DataFrame([row]).to_csv(COMPARE_CSV, index=False)
else:
    df = pd.read_csv(COMPARE_CSV)
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)   
    df.to_csv(COMPARE_CSV, index=False)
print("Appended MLP results to", COMPARE_CSV)

Appended MLP results to ../artifacts\model_comparison.csv


In [17]:
# Run this in the same environment where MLP was evaluated (uses same X_hold, y_hold)
import joblib, json, numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import sys
sys.path.append("..")
from src.transforms import select_cols_top7
MODEL_XGB = "../artifacts/final_lightweight_model_k7_clean.joblib"   # adjust if different
# load X_hold, y_hold from your MLP notebook or recompute same split
import pandas as pd
df = pd.read_csv("../data/heart_uci.csv")
df['num'] = (df['num'] > 0).astype(int)
X = df.drop(columns=['num'])
y = df['num'].values
from sklearn.model_selection import train_test_split
X_train, X_hold, y_train, y_hold = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# load saved XGB pipeline (calibrated wrapper expected)
model_xgb = joblib.load(MODEL_XGB)
print("Loaded XGB model type:", type(model_xgb))

# predict probabilities using the saved pipeline (it includes preprocessing)
probs_xgb = model_xgb.predict_proba(X_hold)[:,1]
roc_xgb = roc_auc_score(y_hold, probs_xgb)
ap_xgb = average_precision_score(y_hold, probs_xgb)
brier_xgb = brier_score_loss(y_hold, probs_xgb)
print("XGB on same holdout -> ROC-AUC:", roc_xgb, "AP:", ap_xgb, "Brier:", brier_xgb)

# compare to MLP numbers you reported (paste them in)
# mlp: roc=0.9226446676, ap=0.936821, brier=0.118171


Loaded XGB model type: <class 'sklearn.calibration.CalibratedClassifierCV'>
XGB on same holdout -> ROC-AUC: 0.9246174079387852 AP: 0.9288074235651942 Brier: 0.11038611136974262
